# Projet — Exploitation du dateset CL-Drive

## Estimation de la charge cognitive du conducteur

Ce TP vient après les TD1, TD2, TD3 et TD4.

Les TD ont déjà permis de travailler :

- la compréhension du papier CL-Drive ;
- le protocole expérimental ;
- la segmentation en fenêtres de 10 s ;
- le prétraitement EEG ;
- l'extraction des features EEG.

Le point de départ du TP est donc le dossier généré à la fin du TD4 :

```text
EEG_Features_10s/
```

Ce TP ne revient pas sur le calcul des features. Il exploite les features déjà extraites pour construire un pipeline d'apprentissage automatique.

## Objectif du TP

Construire un pipeline complet :

```text
EEG_Features_10s
→ Normalized_Features_10s/EEG
→ Normalized_Features_10s_With_Label/EEG
→ Dataset EEG supervisé
→ Classification de la charge cognitive
→ Évaluation
→ Interprétation
```

Dans un premier temps, on se limite à l'EEG uniquement.

La multimodalité, c'est-à-dire l'ajout de ECG, EDA et Gaze, sera proposée uniquement comme extension à la fin du sujet.

## 1. Structure attendue des dossiers

Avant de commencer, le dossier de travail doit contenir au minimum :

```text
Data/
├── EEG/ID_x
│   ├── ... fichiers level_1, level_2, ..., level_9
│   ├── ... fichiers baseline
│   └── ... fichiers filtered_*
│
├── EEG_Features_10s/
│   ├── ID1_EEG_features.csv
│   ├── ID2_EEG_features.csv
│   └── ...
│
├── Labels/
│   ├── ID1.csv
│   ├── ID2.csv
│   └── ...
```

Le TP va générer deux nouveaux dossiers :

```text
Data/
├── Normalized_Features_10s/
│   └── EEG/
│       ├── norm_ID1_EEG_features.csv
│       ├── norm_ID2_EEG_features.csv
│       └── ...
│
├── Normalized_Features_10s_With_Label/(avec colonnes Level et Label)
│   └── EEG/
│       ├── norm_ID1_EEG_features.csv
│       ├── norm_ID2_EEG_features.csv
│       └── ...
```

## Question

Pourquoi ne faut-il pas entraîner directement les modèles sur les fichiers `EEG_Features_10s`  ?

### Réponse

Les features brutes ne sont pas comparables entre sujets : les amplitudes EEG varient fortement d'une personne à l'autre selon l'anatomie et la qualité de contact des électrodes. Sans normalisation, le modèle apprendrait des différences inter-individuelles plutôt que des patterns physiologiques liés à la charge cognitive.

In [2]:
from pathlib import Path

# Adaptation à notre organisation issue du TD4
BASE_PATH = Path("CL-Drive")

EEG_FEATURE_DIR = BASE_PATH / "Features"
LABEL_DIR = BASE_PATH / "Labels"
NORMALIZED_ROOT = BASE_PATH / "Normalized_Features_10s"
NORMALIZED_EEG_DIR = NORMALIZED_ROOT / "EEG"
LABELED_ROOT = BASE_PATH / "Normalized_Features_10s_With_Label"
LABELED_EEG_DIR = LABELED_ROOT / "EEG"

NORMALIZED_EEG_DIR.mkdir(parents=True, exist_ok=True)
LABELED_EEG_DIR.mkdir(parents=True, exist_ok=True)

METADATA_COLUMNS = ["Participant", "File", "Window", "Start_Time", "End_Time", "Channel"]

### Choix d'organisation du pipeline

Les chemins sont définis de manière relative à `CL-Drive/` afin de rendre le notebook portable. Les dossiers `Normalized_Features_10s/EEG` et `Normalized_Features_10s_With_Label/EEG` sont créés automatiquement s'ils n'existent pas. La liste `METADATA_COLUMNS` centralise les noms des colonnes non-numériques à exclure de toute normalisation, ce qui évite de les modifier à la main dans chaque fonction.

## 2. Normalisation des features EEG

À la fin du TD4, chaque fichier CSV contient des features EEG calculées sur des fenêtres de 10 secondes.

La normalisation doit suivre deux étapes :

### Étape 1 — Normalisation par la baseline du sujet

Pour chaque sujet, les fichiers de baseline servent à calculer une valeur moyenne de référence pour chaque feature :

$$
\mu_{baseline}^{(s,f)} = \frac{1}{N}\sum_{i=1}^{N} x_i^{(s,f)}
$$

où :

- $s$ désigne le sujet ;
- $f$ désigne la feature ;
- $x_i^{(s,f)}$ désigne la valeur de la feature pendant la baseline.

Chaque valeur de feature dans les fichiers de tâche est ensuite divisée par la moyenne de baseline correspondante :

$$
x_{norm}^{(s,f)} = \frac{ x^{(s,f)} }{ \mu_{baseline}^{(s,f)} }
$$

### Étape 2 — Standardisation z-score

On applique ensuite une standardisation :

$$
z = \frac{x - \mu}{\sigma}
$$

Cela permet d’obtenir des features centrées et réduites. Cette étape devra toutefois être réalisée plus loin dans le pipeline, après la séparation des données entre les ensembles d’entraînement et de test (voir section 8 ci-dessous).

## Question

Quel est l'intérêt de la normalisation par baseline dans des signaux physiologiques ?

### Réponse

Les signaux physiologiques ont des amplitudes très différentes d'un individu à l'autre. La baseline, enregistrée au repos avant les scénarios, fournit une référence personnelle pour chaque sujet. Diviser les features de tâche par cette référence supprime les différences inter-individuelles et ne conserve que les variations relatives induites par la charge cognitive.

In [3]:
import numpy as np
import pandas as pd

def get_feature_columns(df, metadata_columns=METADATA_COLUMNS):
    """
    Retourne les colonnes numériques correspondant aux features.
    Les colonnes de métadonnées ne doivent pas être normalisées.
    """
    # Sélectionne les colonnes numériques
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    # Retire les métadonnées pour ne laisser que les vraies features
    return [col for col in numeric_cols if col not in metadata_columns]


def compute_baseline_averages(feature_dir):
    """
    Calcule, pour chaque Participant, la moyenne de baseline de chaque feature.
    """
    baseline_avgs = {}
    
    # Dans le TD4, nous avons généré un seul grand fichier global :
    global_file = feature_dir / "CL_Drive_All_Features.csv"
    
    if global_file.exists():
        df = pd.read_csv(global_file)
        
        # Garder uniquement les lignes des fichiers "baseline"
        df_baseline = df[df['File'].str.contains('baseline', case=False, na=False)]
        features_cols = get_feature_columns(df)
        
        # Grouper par Participant et calculer la moyenne
        baseline_avgs = df_baseline.groupby('Participant')[features_cols].mean().to_dict(orient='index')
        
    return baseline_avgs


def normalize_by_baseline(df, participant_id, baseline_avgs):
    """
    Divise chaque feature par sa moyenne de baseline pour le sujet considéré.
    """
    df_norm = df.copy()
    feature_cols = get_feature_columns(df)
    
    if participant_id in baseline_avgs:
        avgs = baseline_avgs[participant_id]
        
        # On divise la colonne entière par la valeur scalaire de la moyenne de baseline
        for feat in feature_cols:
            if avgs[feat] != 0:
                df_norm[feat] = df_norm[feat] / avgs[feat]
            else:
                df_norm[feat] = 0.0
                
    return df_norm

def run_eeg_normalization():
    """
    Génère les fichiers du dossier : Normalized_Features_10s/EEG
    à partir du fichier global issu du TD4.
    """
    # 1. Calculer les profils baseline
    baseline_avgs = compute_baseline_averages(EEG_FEATURE_DIR)
    
    global_file = EEG_FEATURE_DIR / "CL_Drive_All_Features.csv"
    if not global_file.exists():
        print(f"Erreur : le fichier {global_file} n'existe pas.")
        return
        
    df = pd.read_csv(global_file)
    
    # 2. Exclure les signaux de baseline pour ne normaliser que les tests (level_1 à 9)
    df_task = df[~df['File'].str.contains('baseline', case=False, na=False)]
    
    count = 0
    # 3. Parcourir et formater la sortie par participant
    for participant_id, group_df in df_task.groupby('Participant'):
        # Normalisation
        df_norm = normalize_by_baseline(group_df, participant_id, baseline_avgs)
        
        # Sauvegarder sous le nom norm_<nom_fichier>.csv dans NORMALIZED_EEG_DIR
        out_file = NORMALIZED_EEG_DIR / f"norm_{participant_id}_EEG_features.csv"
        df_norm.to_csv(out_file, index=False)
        count += 1
        
    print(f"[{count} fichiers] Normalisation intra-sujet terminée dans : {NORMALIZED_EEG_DIR}")

In [4]:
run_eeg_normalization()

[21 fichiers] Normalisation intra-sujet terminée dans : CL-Drive\Normalized_Features_10s\EEG


### Résultat attendu — Normalisation

L'exécution de `run_eeg_normalization()` doit afficher :

```
[N fichiers] Normalisation intra-sujet terminée dans : CL-Drive/Normalized_Features_10s/EEG
```

où `N` correspond au nombre de participants dans le dataset (typiquement 21 dans CL-Drive). Chaque fichier `norm_<ID>_EEG_features.csv` contient les mêmes colonnes que le fichier source, mais les valeurs des features numériques ont été divisées par la moyenne de baseline du sujet correspondant. Les métadonnées (`Participant`, `File`, `Window`, etc.) sont conservées inchangées.

## 3. Vérification du dossier `Normalized_Features_10s/EEG`

Après exécution de la normalisation, vérifiez que le dossier contient bien des fichiers `norm_*.csv`.

## Question

Pourquoi les fichiers de baseline ne sont-ils pas copiés dans le dossier normalisé final ?

### Réponse

La baseline sert uniquement à calculer la référence de normalisation de chaque sujet. Une fois cette étape effectuée, les fenêtres de baseline ne correspondent à aucun scénario de conduite et n'ont pas de label de charge cognitive associé, donc les inclure dans le dataset d'apprentissage n'aurait pas de sens.

In [5]:
# Vérification du dossier Normalized_Features_10s/EEG
import os
import pandas as pd

# On liste tous les fichiers CSV normalisés
normalized_files = list(NORMALIZED_EEG_DIR.glob("norm_*.csv"))
print(f"Nombre de fichiers normalisés trouvés : {len(normalized_files)}")

if len(normalized_files) > 0:
    print("\nExemple de fichiers générés :")
    for f in normalized_files[:5]:
        print(f" - {f.name}")
        
    # On charge le premier fichier pour un aperçu
    sample_feat = pd.read_csv(normalized_files[0])
    print(f"\nAperçu des données du premier fichier ({normalized_files[0].name}) | Taille : {sample_feat.shape} :")
    display(sample_feat.head())
else:
    print("\nAucun fichier trouvé. Veuillez vérifier l'étape de normalisation.")

Nombre de fichiers normalisés trouvés : 21

Exemple de fichiers générés :
 - norm_1030_EEG_features.csv
 - norm_1105_EEG_features.csv
 - norm_1106_EEG_features.csv
 - norm_1241_EEG_features.csv
 - norm_1271_EEG_features.csv

Aperçu des données du premier fichier (norm_1030_EEG_features.csv) | Taille : (640, 46) :


,Window,Start_Time,End_Time,Channel,Participant,File,delta_psd_sum,delta_psd_mean,delta_psd_max,delta_psd_min,...,hjorth_mobility,hjorth_complexity,lempel_ziv,higuchi_fd,time_mean,time_min,time_max,time_median,time_variance,time_std
0,1,-1.128732e-01,-3.431876e-09,TP9,1030,filtered_eeg_data_level_1.csv,0.886824,0.886824,0.815122,1.219379,...,0.771842,1.129275,0.949635,0.958203,30.157280,1.401581,1.212308,2.481900,1.093185,1.215878
1,1,-1.128732e-01,-3.431876e-09,AF7,1030,filtered_eeg_data_level_1.csv,0.146345,0.146345,0.184323,0.112970,...,1.057095,0.792880,1.042457,0.996892,-66.412450,0.448435,0.551170,-0.147620,0.198759,0.518449
2,1,-1.128732e-01,-3.431876e-09,AF8,1030,filtered_eeg_data_level_1.csv,0.295196,0.295196,0.224225,0.324547,...,1.017698,0.840964,0.999616,0.996055,60.457739,0.488132,1.030500,0.201313,0.387701,0.724088
3,1,-1.128732e-01,-3.431876e-09,TP10,1030,filtered_eeg_data_level_1.csv,0.727522,0.727522,0.692456,0.770801,...,0.880575,1.036301,1.078158,0.989260,-37.028533,1.469056,1.046191,1.138174,0.939070,1.126918
4,2,-3.409591e-09,8.444117e-13,TP9,1030,filtered_eeg_data_level_1.csv,1.558226,1.558226,1.118394,2.524230,...,0.669184,1.226627,0.892515,0.911252,12.053188,1.819707,1.137669,1.843251,1.659230,1.497947


### Analyse — Vérification de la normalisation

L'exécution produit 21 fichiers normalisés, un par participant (ex. norm_1030_EEG_features.csv, norm_1105_EEG_features.csv…).
Le premier fichier inspecté (norm_1030_EEG_features.csv) présente une taille de 640 lignes × 46 colonnes, correspondant à 4 canaux EEG (TP9, AF7, AF8, TP10) × 160 fenêtres de 10 secondes.
Les valeurs des features normalisées sont cohérentes avec une division par la moyenne de baseline : elles varient autour de 1 pour les features spectrales stables (ex. delta_psd_sum ≈ 0.89 pour TP9), mais peuvent s'en éloigner significativement selon le canal et la feature (ex. delta_psd_sum ≈ 0.15 pour AF7), ce qui reflète des différences réelles de dynamique EEG entre canaux.

**Point d'attention :** les valeurs de Start_Time et End_Time pour la première fenêtre sont quasi nulles ou négatives (ex. -1.13e-01, -3.43e-09), ce qui semble aberrant. Ces colonnes sont des métadonnées non utilisées dans la classification, mais il convient de vérifier leur origine si elles doivent être interprétées.

## 4. Ajout des colonnes `Level` et `Label`

Les fichiers normalisés ne contiennent pas encore la cible d'apprentissage.

Il faut maintenant associer chaque fenêtre de 10 secondes à son score PAAS.

Les labels sont stockés dans le dossier :

```text
Labels/
```

Chaque fichier de labels correspond à un sujet, par exemple :

```text
Labels/ID1.csv
Labels/ID2.csv
...
```

Dans ces fichiers, on suppose une structure du type :

| time | lvl_1 | lvl_2 | ... | lvl_9 |
|---:|---:|---:|---|---:|
| 10 | 2 | 3 | ... | 5 |
| 20 | 2 | 4 | ... | 6 |
| ... | ... | ... | ... | ... |

Pour une fenêtre d'indice `Window`, le temps associé est :

$$
time = (Window + 1) \times 10
$$

Le niveau du scénario est extrait du nom du fichier avec une expression régulière :

```text
level_1 → Level = 1
level_2 → Level = 2
...
level_9 → Level = 9
```

Le score PAAS est ensuite récupéré dans la colonne :

```text
lvl_<Level>
```

Exemple : si `Level = 4`, on lit la colonne `lvl_4`.


In [6]:
import re

def extract_level_from_filename(file_name):
    """
    Extrait le niveau de scénario à partir du nom de fichier.
    Exemple : "filtered_eeg_data_level_3.csv" → 3
    """
    match = re.search(r'level_(\d+)', file_name)
    if match:
        return int(match.group(1))
    return None

def get_label_for_row(row, labels_df):
    """
    Retourne le score PAAS correspondant à une ligne de features.
    """
    time_stamp = int((row['Window']) * 10)
    level = row['Level']
    
    if pd.isna(level):
        return None
        
    label_col = f'lvl_{int(level)}'
    
    if label_col not in labels_df.columns:
        return None
        
    match = labels_df[labels_df['time'] == time_stamp]
    
    if not match.empty:
        return match[label_col].iloc[0]
    return None

def attach_labels_eeg():
    """
    Génère les fichiers du dossier : Normalized_Features_10s_With_Label/EEG
    """
    normalized_files = list(NORMALIZED_EEG_DIR.glob("norm_*.csv"))
    count = 0
    
    for file_path in normalized_files:
        df = pd.read_csv(file_path)
        participant_id = df['Participant'].iloc[0]
        
        label_file = LABEL_DIR / f"{participant_id}.csv"
        
        if not label_file.exists():
            print(f"Erreur : le fichier label {label_file} n'existe pas.")
            continue
            
        labels_df = pd.read_csv(label_file)
        
        df['Level'] = df['File'].apply(extract_level_from_filename)
        df = df.dropna(subset=['Level'])
        
        df['Label'] = df.apply(lambda row: get_label_for_row(row, labels_df), axis=1)
        df = df.dropna(subset=['Label'])
        
        df['Level'] = df['Level'].astype(int)
        
        out_file = LABELED_EEG_DIR / file_path.name
        df.to_csv(out_file, index=False)
        count += 1
        
    print(f"[{count} fichiers] Ajout des labels terminé dans : {LABELED_EEG_DIR}")

In [7]:
attach_labels_eeg()

[21 fichiers] Ajout des labels terminé dans : CL-Drive\Normalized_Features_10s_With_Label\EEG


### Résultat attendu — Ajout des labels

L'exécution de `attach_labels_eeg()` doit afficher :

```
[N fichiers] Ajout des labels terminé dans : CL-Drive/Normalized_Features_10s_With_Label/EEG
```

Chaque fichier dispose maintenant des colonnes `Level` (entier de 1 à 9) et `Label` (score PAAS brut). Les fenêtres pour lesquelles aucun label n'a pu être trouvé dans le fichier PAAS sont supprimées (`dropna`). Cela peut se produire si la durée du scénario enregistrée dans les features dépasse la durée couverte par le fichier de labels.

## 5. Vérification du dossier `Normalized_Features_10s_With_Label/EEG`

Le dossier final doit contenir des fichiers CSV avec au moins :

- les métadonnées : `Participant`, `File`, `Window`, `Channel`, `Start_Time` et `End_Time` ;
- les features EEG normalisées ;
- la colonne `Level` ;
- la colonne `Label`.

## Question

Quelle est la différence entre `Level` et `Label` dans ce TP ? Pourquoi faut-il ajouter à la fois `Level` et `Label` ?

### Réponse

`Level` est le numéro du scénario de conduite (1 à 9), défini par le protocole expérimental. `Label` est la classe binaire de charge cognitive attribuée à ce niveau (0 = faible, 1 = élevée), issue de la binarisation du score PAAS. Conserver les deux permet d'analyser la distribution des labels par scénario et de retrouver le contexte expérimental d'une fenêtre si besoin.

In [8]:
# Vérification du dossier Normalized_Features_10s_With_Label/EEG
import os
import pandas as pd

labeled_files = list(LABELED_EEG_DIR.glob("norm_*.csv"))
print(f"Nombre de fichiers labellisés trouvés : {len(labeled_files)}")

if len(labeled_files) > 0:
    sample_labeled = pd.read_csv(labeled_files[0])
    print(f"\nAperçu des données du premier fichier ({labeled_files[0].name}) | Taille : {sample_labeled.shape} :")
    # On affiche specifiquement les métadonnées et la fin du tableau (là où il y a les labels)
    display(sample_labeled[['Window', 'File', 'Level', 'Label']].head())
else:
    print("\nAucun fichier trouvé.")

Nombre de fichiers labellisés trouvés : 21

Aperçu des données du premier fichier (norm_1030_EEG_features.csv) | Taille : (640, 48) :


,Window,File,Level,Label
0,1,filtered_eeg_data_level_1.csv,1,2.0
1,1,filtered_eeg_data_level_1.csv,1,2.0
2,1,filtered_eeg_data_level_1.csv,1,2.0
3,1,filtered_eeg_data_level_1.csv,1,2.0
4,2,filtered_eeg_data_level_1.csv,1,2.0


### Analyse — Vérification des labels

L'exécution produit 21 fichiers labellisés, un par participant. Le fichier norm_1030_EEG_features.csv passe de 46 à 48 colonnes par rapport à la version normalisée, confirmant l'ajout des deux colonnes Level et Label.
Sur les 5 premières lignes affichées, toutes correspondent au scénario level_1, avec Level = 1 et Label = 2.0. Plusieurs fenêtres différentes (Window 1 et 2) partagent le même score PAAS, ce qui est attendu : le score PAAS est attribué par scénario et non par fenêtre individuelle.

**Point d'attention :** Label prend ici la valeur 2.0 (flottant), non entier. Il faut s'assurer que cette valeur est bien convertie en entier avant la binarisation en Label_Binary.

## 6. Construction du dataset EEG supervisé

Une fois les fichiers normalisés et labellisés générés, on peut les concaténer pour construire un tableau unique.

Chaque ligne représente une fenêtre EEG de 10 secondes pour un canal.

On construit ensuite deux problèmes possibles :

### Classification binaire

| Score PAAS | Classe |
|---:|---|
| 1 à 4 | faible |
| 5 à 9 | élevée |

### Classification ternaire, extension

| Score PAAS | Classe |
|---:|---|
| 1 à 3 | faible |
| 4 à 6 | moyenne |
| 7 à 9 | élevée |

Dans ce TP, l'objectif principal est la classification binaire.

In [9]:
def load_labeled_eeg_dataset():
    """
    Concatène tous les fichiers CSV du dossier Normalized_Features_10s_With_Label/EEG.
    """
    labeled_files = list(LABELED_EEG_DIR.glob("*.csv"))
    
    if not labeled_files:
        print("Aucun fichier trouvé dans LABELED_EEG_DIR.")
        return pd.DataFrame()
        
    dfs = [pd.read_csv(f) for f in labeled_files]
    return pd.concat(dfs, ignore_index=True)

df = load_labeled_eeg_dataset()
print("Taille globale du dataset :", df.shape)
display(df.head())

Taille globale du dataset : (12552, 48)


,Window,Start_Time,End_Time,Channel,Participant,File,delta_psd_sum,delta_psd_mean,delta_psd_max,delta_psd_min,...,lempel_ziv,higuchi_fd,time_mean,time_min,time_max,time_median,time_variance,time_std,Level,Label
0,1,-1.128732e-01,-3.431876e-09,TP9,1030,filtered_eeg_data_level_1.csv,0.886824,0.886824,0.815122,1.219379,...,0.949635,0.958203,30.157280,1.401581,1.212308,2.481900,1.093185,1.215878,1,2.0
1,1,-1.128732e-01,-3.431876e-09,AF7,1030,filtered_eeg_data_level_1.csv,0.146345,0.146345,0.184323,0.112970,...,1.042457,0.996892,-66.412450,0.448435,0.551170,-0.147620,0.198759,0.518449,1,2.0
2,1,-1.128732e-01,-3.431876e-09,AF8,1030,filtered_eeg_data_level_1.csv,0.295196,0.295196,0.224225,0.324547,...,0.999616,0.996055,60.457739,0.488132,1.030500,0.201313,0.387701,0.724088,1,2.0
3,1,-1.128732e-01,-3.431876e-09,TP10,1030,filtered_eeg_data_level_1.csv,0.727522,0.727522,0.692456,0.770801,...,1.078158,0.989260,-37.028533,1.469056,1.046191,1.138174,0.939070,1.126918,1,2.0
4,2,-3.409591e-09,8.444117e-13,TP9,1030,filtered_eeg_data_level_1.csv,1.558226,1.558226,1.118394,2.524230,...,0.892515,0.911252,12.053188,1.819707,1.137669,1.843251,1.659230,1.497947,1,2.0


### Analyse — Dataset global

Le dataset concaténé contient 12 552 lignes × 48 colonnes, correspondant à l'ensemble des fenêtres EEG de tous les participants, tous scénarios confondus. Cela est cohérent avec : 21 participants × 4 canaux × 160 fenêtres ≈ 13 440 lignes attendues. L'écart s'explique par des fenêtres supprimées lors du dropna à l'étape d'ajout des labels.
Chaque ligne est une observation indépendante (une fenêtre de 10 s × un canal). Les colonnes Level et Label figurent bien en fin de tableau, confirmant la bonne concaténation des fichiers labellisés.

In [10]:
import numpy as np

# Création des cibles de classification binaire (0 = faible, 1 = élevée)
# Score PAAS 1 à 4 -> faible (0), 5 à 9 -> élevée (1)
df["Label_Binary"] = np.where(df["Label"] <= 4, 0, 1)

# Extension ternaire éventuelle (0 = faible, 1 = moyenne, 2 = élevée)
# Score PAAS 1 à 3 -> faible (0), 4 à 6 -> moyenne (1), 7 à 9 -> élevée (2)
conditions = [
    df["Label"] <= 3,
    (df["Label"] >= 4) & (df["Label"] <= 6),
    df["Label"] >= 7
]
choices = [0, 1, 2]
df["Label_Ternary"] = np.select(conditions, choices, default=-1)

print("Distribution de Label_Binary :")
print(df["Label_Binary"].value_counts())

print("\nDistribution de Label_Ternary :")
print(df["Label_Ternary"].value_counts())

Distribution de Label_Binary :
Label_Binary
1    7248
0    5304
Name: count, dtype: int64

Distribution de Label_Ternary :
Label_Ternary
1    6600
0    3320
2    2632
Name: count, dtype: int64


### Analyse — Distribution des classes

Classification binaire : la classe 1 (charge élevée) regroupe 7 248 exemples contre 5 304 pour la classe 0 (charge faible), soit respectivement 57,7 % et 42,3 %. Le déséquilibre est modéré mais réel — il convient donc de privilégier le F1-score pondéré plutôt que l'accuracy comme métrique principale d'évaluation.
Classification ternaire : la distribution est plus déséquilibrée, avec 6 600 exemples de classe 1 (charge moyenne, 52,6 %), 3 320 de classe 0 (charge faible, 26,5 %) et 2 632 de classe 2 (charge élevée, 21,0 %). Ce déséquilibre marqué renforce le choix de se concentrer sur la classification binaire dans ce TP.

## 7. Préparation de la matrice d'apprentissage

On doit séparer :

- les métadonnées ;
- les features numériques EEG ;
- la cible d'apprentissage.

## Question

Pourquoi ne faut-il pas inclure `Participant`, `File`, `Window`, `Level` ou `Label` dans les features du modèle ?

### Réponse

Ces colonnes sont des identifiants ou la variable cible, pas des mesures physiologiques. Les inclure comme features apprendrait au modèle des associations parasites, par exemple que tel sujet ou tel scénario est systématiquement associé à une certaine classe, au lieu de patterns généralisables à de nouveaux conducteurs.

In [11]:
# Colonnes à exclure de la matrice de features (métadonnées et cibles)
columns_to_exclude = ["Participant", "File", "Window", "Start_Time", "End_Time", "Channel", "Level", "Label", "Label_Binary", "Label_Ternary"]

# Sélection des features : on prend toutes les colonnes sauf celles à exclure
feature_cols = [col for col in df.columns if col not in columns_to_exclude]

X = df[feature_cols].copy()
y = df["Label_Binary"].copy() # On sélectionne la classification binaire comme cible principale
groups = df["Participant"].copy() # Le groupe participant sera utile pour la CV LOSO

print("Nombre d'exemples :", X.shape[0])
print("Nombre de features :", X.shape[1])
print("Exemples de features :", feature_cols[:10])

# Aperçu de la matrice finale
display(X.head())

Nombre d'exemples : 12552
Nombre de features : 40
Exemples de features : ['delta_psd_sum', 'delta_psd_mean', 'delta_psd_max', 'delta_psd_min', 'delta_psd_median', 'theta_psd_sum', 'theta_psd_mean', 'theta_psd_max', 'theta_psd_min', 'theta_psd_median']


,delta_psd_sum,delta_psd_mean,delta_psd_max,delta_psd_min,delta_psd_median,theta_psd_sum,theta_psd_mean,theta_psd_max,theta_psd_min,theta_psd_median,...,hjorth_mobility,hjorth_complexity,lempel_ziv,higuchi_fd,time_mean,time_min,time_max,time_median,time_variance,time_std
0,0.886824,0.886824,0.815122,1.219379,0.832148,1.511590,1.511590,1.044128,1.748544,1.937953,...,0.771842,1.129275,0.949635,0.958203,30.157280,1.401581,1.212308,2.481900,1.093185,1.215878
1,0.146345,0.146345,0.184323,0.112970,0.124583,0.144689,0.144689,0.079266,0.149622,0.204903,...,1.057095,0.792880,1.042457,0.996892,-66.412450,0.448435,0.551170,-0.147620,0.198759,0.518449
2,0.295196,0.295196,0.224225,0.324547,0.354788,0.383584,0.383584,0.401994,0.217929,0.396908,...,1.017698,0.840964,0.999616,0.996055,60.457739,0.488132,1.030500,0.201313,0.387701,0.724088
3,0.727522,0.727522,0.692456,0.770801,0.768999,1.433151,1.433151,1.173085,1.925809,1.668571,...,0.880575,1.036301,1.078158,0.989260,-37.028533,1.469056,1.046191,1.138174,0.939070,1.126918
4,1.558226,1.558226,1.118394,2.524230,1.634445,3.066830,3.066830,2.221674,4.571109,4.116519,...,0.669184,1.226627,0.892515,0.911252,12.053188,1.819707,1.137669,1.843251,1.659230,1.497947


### Analyse — Matrice d'apprentissage

La matrice finale contient 12 552 exemples × 40 features, après exclusion des 8 colonnes de métadonnées et de cibles (Participant, File, Window, Start_Time, End_Time, Channel, Level, Label, Label_Binary, Label_Ternary).
Les 40 features retenues couvrent plusieurs familles de descripteurs EEG : puissances spectrales par bande (delta, theta…), paramètres de Hjorth (hjorth_mobility, hjorth_complexity), complexité du signal (lempel_ziv, higuchi_fd) et statistiques temporelles (time_mean, time_min, time_max, time_median, time_variance, time_std).
Les valeurs sont cohérentes avec la normalisation par baseline : la plupart des features spectrales sont proches de 1.

## 8. Classification EEG — premiers modèles

On teste plusieurs modèles classiques :

- LDA ;
- SVM ;
- Random Forest ;
- KNN ;
- Naive Bayes ;
- Decision Tree ;
- AdaBoost ;
- MLP.

La normalisation `StandardScaler` est placée dans le `sklearn.pipeline.Pipeline` pour éviter une fuite de données entre apprentissage et test. Il faut ajuster le `StandardScaler` uniquement sur les données d’entraînement :

`scaler.fit_transform(X_train)`

Puis appliquer la transformation aux données de test avec :

`scaler.transform(X_test)`

In [12]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

# Import des classifieurs
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier

# Définition des modèles de base
models = {
    "LDA": LinearDiscriminantAnalysis(),
    "SVM": SVC(kernel='rbf', random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Naive Bayes": GaussianNB(),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "AdaBoost": AdaBoostClassifier(random_state=42),
    "MLP": MLPClassifier(hidden_layer_sizes=(100,), max_iter=1000, random_state=42)
}

# Création des pipelines avec StandardScaler
pipelines = {name: Pipeline([('scaler', StandardScaler()), ('clf', model)]) 
             for name, model in models.items()}

# --- Test rapide pour vérifier le bon fonctionnement de chaque pipeline ---
# On fait une simple séparation 80% train / 20% test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("--- Résultats préliminaires sur un test 80/20 (hors CV) ---\n")
for name, pipeline in pipelines.items():
    # Entraînement complet du pipeline (fit ajuste le scaler ET le classifieur)
    pipeline.fit(X_train, y_train)
    
    # Prédiction (transforme automatiquement X_test avant prédiction)
    y_pred = pipeline.predict(X_test)
    
    # Évaluation
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='macro')
    
    print(f"{name:>15} -> Accuracy : {acc:.3f} | F1-Score (macro) : {f1:.3f}")

--- Résultats préliminaires sur un test 80/20 (hors CV) ---

            LDA -> Accuracy : 0.597 | F1-Score (macro) : 0.511
            SVM -> Accuracy : 0.684 | F1-Score (macro) : 0.653
  Random Forest -> Accuracy : 0.764 | F1-Score (macro) : 0.751
            KNN -> Accuracy : 0.695 | F1-Score (macro) : 0.686
    Naive Bayes -> Accuracy : 0.465 | F1-Score (macro) : 0.410
  Decision Tree -> Accuracy : 0.644 | F1-Score (macro) : 0.636


c:\Anaconda3\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:519: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(


       AdaBoost -> Accuracy : 0.667 | F1-Score (macro) : 0.648
            MLP -> Accuracy : 0.718 | F1-Score (macro) : 0.708


### Résultats — Test préliminaire 80/20

Ce test rapide sert uniquement à vérifier le bon fonctionnement des pipelines, pas à comparer rigoureusement les modèles (la validation croisée en section 9 est plus fiable).

| Modèle        | Accuracy | F1-Score (macro) |
|:--------------|:--------:|:----------------:|
| LDA           | 0.597    | 0.511            |
| SVM           | 0.684    | 0.653            |
| Random Forest | 0.764    | 0.751            |
| KNN           | 0.695    | 0.686            |
| Naive Bayes   | 0.465    | 0.410            |
| Decision Tree | 0.644    | 0.636            |
| AdaBoost      | 0.667    | 0.648            |
| MLP           | 0.718    | 0.708            |

Random Forest obtient les meilleures performances (Accuracy : 0.764, F1 : 0.751), suivi de MLP et KNN. Naive Bayes est nettement en retrait (F1 : 0.410), ce qui suggère que l'hypothèse d'indépendance des features n'est pas adaptée aux données EEG.

## 9. Évaluation par validation croisée et par sujet

Deux évaluations sont demandées :

### 10-fold cross-validation

Les segments sont répartis en 10 folds stratifiés. Cette évaluation est utile pour comparer les modèles, mais elle peut mélanger les sujets entre apprentissage et test.

### Leave-One-Subject-Out, LOSO

Un sujet est laissé de côté pour le test, tandis que le modèle est entraîné sur les autres sujets. Cette stratégie d’évaluation est plus réaliste, car elle permet de tester la capacité de généralisation du modèle sur un conducteur jamais vu auparavant. L’opération est ensuite répétée sur l’ensemble des sujets disponibles afin d’obtenir une évaluation plus robuste.

## Question

Pourquoi le LOSO est-il souvent plus difficile que le 10-fold classique ?

### Réponse

En 10-fold, des fenêtres du même sujet se retrouvent à la fois dans le train et dans le test, ce qui facilite artificiellement la prédiction car les patterns individuels sont déjà connus. En LOSO, le sujet de test est entièrement absent de l'entraînement, ce qui exige une vraie généralisation à un conducteur jamais vu.

In [13]:
import numpy as np
from sklearn.model_selection import StratifiedKFold, LeaveOneGroupOut, cross_validate

# 1. Configuration des stratégies de validation croisée
cv_10fold = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
cv_loso = LeaveOneGroupOut()

# Métriques à calculer
scoring = ['accuracy', 'f1_macro']

print("="*50)
print("ÉVALUATION 1 : 10-Fold Stratifié")
print("="*50)
for name, pipeline in pipelines.items():
    results_10f = cross_validate(pipeline, X, y, cv=cv_10fold, scoring=scoring)
    acc_10f = np.mean(results_10f['test_accuracy'])
    f1_10f = np.mean(results_10f['test_f1_macro'])
    print(f"{name:>15} -> Accuracy : {acc_10f:.3f} | F1-Score : {f1_10f:.3f}")

print("\n" + "="*50)
print("ÉVALUATION 2 : Leave-One-Subject-Out (LOSO)")
print("="*50)
for name, pipeline in pipelines.items():
    # LOSO nécessite de préciser le paramètre `groups`
    results_loso = cross_validate(pipeline, X, y, groups=groups, cv=cv_loso, scoring=scoring)
    acc_loso = np.mean(results_loso['test_accuracy'])
    f1_loso = np.mean(results_loso['test_f1_macro'])
    print(f"{name:>15} -> Accuracy : {acc_loso:.3f} | F1-Score : {f1_loso:.3f}")

ÉVALUATION 1 : 10-Fold Stratifié
            LDA -> Accuracy : 0.601 | F1-Score : 0.517
            SVM -> Accuracy : 0.692 | F1-Score : 0.663
  Random Forest -> Accuracy : 0.758 | F1-Score : 0.744
            KNN -> Accuracy : 0.699 | F1-Score : 0.691
    Naive Bayes -> Accuracy : 0.468 | F1-Score : 0.418
  Decision Tree -> Accuracy : 0.657 | F1-Score : 0.650


c:\Anaconda3\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:519: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
c:\Anaconda3\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:519: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
c:\Anaconda3\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:519: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
c:\Anaconda3\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:519: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
c:\Anaconda3\Lib\site-packages\sklearn\ensemble\_weight_boos

       AdaBoost -> Accuracy : 0.681 | F1-Score : 0.661
            MLP -> Accuracy : 0.731 | F1-Score : 0.720

ÉVALUATION 2 : Leave-One-Subject-Out (LOSO)
            LDA -> Accuracy : 0.557 | F1-Score : 0.451
            SVM -> Accuracy : 0.548 | F1-Score : 0.479
  Random Forest -> Accuracy : 0.552 | F1-Score : 0.474
            KNN -> Accuracy : 0.557 | F1-Score : 0.502
    Naive Bayes -> Accuracy : 0.459 | F1-Score : 0.376
  Decision Tree -> Accuracy : 0.537 | F1-Score : 0.482


c:\Anaconda3\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:519: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
c:\Anaconda3\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:519: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
c:\Anaconda3\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:519: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
c:\Anaconda3\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:519: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
c:\Anaconda3\Lib\site-packages\sklearn\ensemble\_weight_boos

       AdaBoost -> Accuracy : 0.571 | F1-Score : 0.491
            MLP -> Accuracy : 0.575 | F1-Score : 0.505


### Résultats — Évaluation LOSO (Leave-One-Subject-Out)

| Modèle        | Acc. 10-fold | F1 10-fold | Acc. LOSO | F1 LOSO |
|:--------------|:------------:|:----------:|:---------:|:-------:|
| LDA           | 0.601        | 0.517      | 0.557     | 0.451   |
| SVM           | 0.692        | 0.663      | 0.548     | 0.479   |
| Random Forest | 0.758        | 0.744      | 0.552     | 0.474   |
| KNN           | 0.699        | 0.691      | 0.557     | 0.502   |
| Naive Bayes   | 0.468        | 0.418      | 0.459     | 0.376   |
| Decision Tree | 0.657        | 0.650      | 0.537     | 0.482   |
| AdaBoost      | 0.681        | 0.661      | 0.571     | 0.491   |
| MLP           | 0.731        | 0.720      | 0.575     | 0.505   |

Comparaison des stratégies : la chute de performance entre le 10-fold et le LOSO est significative pour tous les modèles. Random Forest, meilleur en 10-fold (F1 : 0.744), tombe à 0.474 en LOSO. MLP résiste légèrement mieux (0.720 → 0.505), tout comme KNN (0.691 → 0.502).
En LOSO, aucun modèle ne dépasse un F1 de 0.505, ce qui confirme que les features EEG seules ne suffisent pas à généraliser sur un conducteur jamais vu. Les performances proches du hasard indiquent une forte variabilité inter-sujets que le pipeline actuel ne parvient pas à absorber.

## 10. Interprétation et discussion

Répondez aux questions suivantes dans le notebook :

1. Quel modèle obtient le meilleur F1-score en 10-fold ?
2. Quel modèle obtient le meilleur F1-score en LOSO ?
3. Les performances chutent-elles en LOSO ? Pourquoi ?
4. Les classes sont-elles équilibrées ?
5. Les résultats obtenus avec EEG seul vous semblent-ils suffisants pour une application réelle ?
6. Quelles limites voyez-vous à l'utilisation des labels subjectifs PAAS ?
7. Quelles améliorations proposeriez-vous ?

### Réponses :

1. Random Forest obtient le meilleur F1-score en 10-fold avec 0.744.

2. MLP obtient le meilleur F1-score en LOSO avec 0.505.

3. Oui, la chute est importante pour tous les modèles, entre 24 et 27 points de F1. En 10-fold, des fenêtres du même sujet se retrouvent dans le train et dans le test, ce qui facilite la prédiction car les patterns individuels sont déjà vus. En LOSO, le sujet de test est entièrement absent de l'entraînement, ce qui exige une généralisation à un conducteur inconnu que les features EEG seules ne permettent pas suffisamment.

4. Non, la distribution est de 57.7 % pour la classe 1 (charge élevée) contre 42 % pour la classe 0 (charge faible). Le déséquilibre est modéré, ce qui justifie l'utilisation du F1 macro plutôt que l'accuracy comme métrique principale.

5. Non. Le meilleur F1 LOSO obtenu est 0.505 (MLP), soit à peine au-dessus du hasard compte tenu du déséquilibre des classes. Un tel niveau de performance n'est pas exploitable dans un système embarqué de sécurité.

6. Le PAAS est rempli après la tâche, ce qui introduit un biais de mémoire. La binarisation par seuil fixe (≤ 4 / ≥ 5) est arbitraire et ne tient pas compte des différences de perception entre individus. Le score peut également être influencé par la fatigue ou l'état émotionnel du participant, indépendamment de la charge cognitive réelle.

7. - Corriger le déséquilibre des classes (57/43)
   - Tester d'autres hyperparamètres pour les modèles les plus prometteurs (Random Forest en 10-fold, MLP en LOSO)
   - Avoir plus de données EEG


## 11. Mini-système d'adaptation

À partir de la prédiction du modèle, on peut simuler une décision d'adaptation.

Exemple :

| Prédiction | Décision |
|---|---|
| charge faible | interface normale |
| charge élevée | simplification de l'interface |
| charge élevée persistante | alerte conducteur |

## Question

Pourquoi faut-il être prudent avant de déclencher une alerte sur une seule prédiction ?

### Réponse

Un classifieur commet des erreurs ponctuelles, et un seul segment de 10 secondes peut être bruité ou atypique. Déclencher une alerte sur une unique prédiction risque de générer de fausses alarmes qui perturbent inutilement le conducteur, ce qui est contre-productif pour la sécurité. En pratique, on attend plusieurs prédictions consécutives de charge élevée avant d'agir.

In [15]:
from collections import Counter

# Nombre de prédictions élevées consécutives avant de déclencher une alerte
CONSECUTIVE_HIGH_THRESHOLD = 3

def decision_system(predictions, consecutive_threshold=CONSECUTIVE_HIGH_THRESHOLD):
    """
    Transforme une séquence de prédictions binaires en décisions d'adaptation.

    Paramètres
    ----------
    predictions : liste ou tableau de prédictions (0 = charge faible, 1 = charge élevée)
    consecutive_threshold : nombre de prédictions élevées consécutives pour déclencher l'alerte

    Retourne
    --------
    Liste de décisions pour chaque fenêtre.
    """
    decisions = []
    consecutive_high = 0

    for pred in predictions:
        if pred == 0:
            consecutive_high = 0
            decisions.append("interface normale")
        else:
            consecutive_high += 1
            if consecutive_high >= consecutive_threshold:
                decisions.append("alerte conducteur")
            else:
                decisions.append("simplification de l'interface")

    return decisions


# --- Démonstration sur les prédictions du meilleur modèle (Random Forest) ---
print("=== Mini-système d'adaptation — démonstration ===\n")

best_model = pipelines["Random Forest"]

# On isole un participant et un seul canal pour illustrer la séquence temporelle
participant_id = df["Participant"].unique()[0]
mask = (df["Participant"] == participant_id) & (df["Channel"] == "TP9")
X_demo = X[mask].reset_index(drop=True)
y_demo = y[mask].reset_index(drop=True)

# Prédictions du modèle pour ce sujet / canal
y_pred_demo = best_model.predict(X_demo)
decisions = decision_system(y_pred_demo, consecutive_threshold=CONSECUTIVE_HIGH_THRESHOLD)

# Affichage des 30 premières fenêtres
n_display = min(30, len(y_pred_demo))
print(f"Participant : {participant_id} | Canal : TP9 | Seuil alerte : {CONSECUTIVE_HIGH_THRESHOLD} consécutives\n")
print(f"{'Fenêtre':>8} | {'Prédit':>13} | {'Réel':>13} | Décision")
print("-" * 72)
for i in range(n_display):
    pred_str = "charge élevée" if y_pred_demo[i] == 1 else "charge faible"
    true_str  = "charge élevée" if y_demo.iloc[i]  == 1 else "charge faible"
    marker = " <-- ALERTE" if decisions[i] == "alerte conducteur" else ""
    print(f"{i+1:>8} | {pred_str:>13} | {true_str:>13} | {decisions[i]}{marker}")

# Résumé global pour ce sujet
print("\n--- Résumé des décisions (sujet complet, canal TP9) ---")
counts = Counter(decisions)
total = len(decisions)
for label in ["interface normale", "simplification de l'interface", "alerte conducteur"]:
    n = counts.get(label, 0)
    print(f"  {label:<38} : {n:>3} fenêtres ({100*n/total:.1f}%)")

=== Mini-système d'adaptation — démonstration ===

Participant : 1030 | Canal : TP9 | Seuil alerte : 3 consécutives

 Fenêtre |        Prédit |          Réel | Décision
------------------------------------------------------------------------
       1 | charge faible | charge faible | interface normale
       2 | charge faible | charge faible | interface normale
       3 | charge faible | charge faible | interface normale
       4 | charge faible | charge faible | interface normale
       5 | charge élevée | charge faible | simplification de l'interface
       6 | charge faible | charge faible | interface normale
       7 | charge faible | charge faible | interface normale
       8 | charge faible | charge faible | interface normale
       9 | charge faible | charge faible | interface normale
      10 | charge faible | charge faible | interface normale
      11 | charge faible | charge faible | interface normale
      12 | charge faible | charge faible | interface normale
      13 | cha

### Analyse — Mini-système d'adaptation

La démonstration (basée sur Random Forest) porte sur le participant 1030, canal TP9, avec un seuil d'alerte fixé à 3 prédictions consécutives de charge élevée.

Sur les 30 premières fenêtres affichées, le modèle prédit correctement une charge faible continue (scénario level_1), avec une seule erreur isolée à la fenêtre 5 (faux positif). Grâce au seuil de 3 consécutives, cette erreur ponctuelle ne déclenche pas d'alerte, ce qui illustre concrètement l'intérêt du mécanisme.

Sur l'ensemble de la séquence (160 fenêtres), la distribution des décisions est la suivante :

| Décision                      | Fenêtres | Proportion |
|:------------------------------|:--------:|:----------:|
| Interface normale             |    46    |   28.8%    |
| Simplification de l'interface |    20    |   12.5%    |
| Alerte conducteur             |    94    |   58.8%    |

La proportion élevée d'alertes (58.8%) est cohérente avec la distribution des classes du dataset : 57.7% des fenêtres sont labellisées charge élevée. Le système d'adaptation reflète donc fidèlement la charge cognitive prédite sur l'ensemble des scénarios du participant.

## 12. Extension optionnelle — vers la multimodalité

Le cœur du TP est volontairement limité à l'EEG.

Une extension possible consiste à reproduire les mêmes étapes pour les autres modalités :

```text
ECG_Features_10s → Normalized_Features_10s/ECG → Normalized_Features_10s_With_Label/ECG
EDA_Features_10s → Normalized_Features_10s/EDA → Normalized_Features_10s_With_Label/EDA
Gaze_Features_10s → Normalized_Features_10s/Gaze → Normalized_Features_10s_With_Label/Gaze
```

Puis à fusionner les features :

```text
EEG + ECG
EEG + EDA
EEG + Gaze
EEG + ECG + EDA + Gaze
```

La fusion la plus simple est une concaténation des colonnes de features pour des fenêtres correspondant au même sujet, au même niveau et au même indice de fenêtre.

## Question

Pourquoi la multimodalité peut-elle améliorer la détection de la charge cognitive ?

### Réponse

Chaque modalité capture un aspect différent de la charge cognitive et présente ses propres limites : l'EEG est précis mais sensible aux artefacts, l'ECG est robuste mais lent, l'EDA est sensible au stress mais peu discriminant sur les niveaux fins. En combinant plusieurs modalités, les erreurs de l'une peuvent être compensées par les autres, ce qui rend la prédiction globalement plus robuste, comme le confirment les résultats du papier.